In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/raw/turkish_law_dataset.csv")

In [4]:
df.head()

,soru,cevap,veri türü,kaynak,context,Score
0,"Anayasa, Türk Vatanı ve Milletinin ebedi varlı...","Anayasa, Türk Vatanı ve Milletinin ebedi varlı...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,8
1,"Anayasa, Türkiye Cumhuriyetinin hangi milliyet...","Anayasa, Türkiye Cumhuriyetinin kurucusu olan ...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,8
2,"Anayasa, Türkiye Cumhuriyetini hangi konumda t...","Anayasa, Türkiye Cumhuriyetini dünya milletler...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,9
3,"Anayasa, Türkiye Cumhuriyetinin hangi hedefler...","Anayasa, Türkiye Cumhuriyetinin ebedi varlığın...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,9
4,"Anayasa, egemenliğin kime ait olduğunu nasıl b...","Anayasa, egemenliğin kayıtsız şartsız Türk Mil...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,10


In [5]:
df.shape

(13954, 6)

In [6]:
df.columns.tolist()

['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'Score']

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13954 entries, 0 to 13953
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   soru       13954 non-null  object
 1   cevap      13954 non-null  object
 2   veri türü  13954 non-null  object
 3   kaynak     13954 non-null  object
 4   context    13954 non-null  object
 5   Score      13954 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 654.2+ KB


In [8]:
df.isnull().sum()

soru         0
cevap        0
veri türü    0
kaynak       0
context      0
Score        0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(247)

In [10]:
df.sample(3, random_state=42)

,soru,cevap,veri türü,kaynak,context,Score
2019,Adlî kontrol hükümlerini yerine getirmeyen şüp...,Adlî kontrol hükümlerini isteyerek yerine geti...,hukuk,Ceza Muhakemesi Kanunu,BİRİNCİ KİTAP\r\nGenel Hükümler\r\nDÖRDÜNCÜ KI...,8
9438,Ceza kanunlarını bilmemek hangi durumlarda maz...,Ceza kanunlarını bilmemek genellikle mazeret s...,hukuk,Türk Ceza Kanunu,Ceza Kanununun amacı\r\nMADDE 1. - (1) Ceza Ka...,8
5709,"Birisi yaşadığı yerin dışında öldüğünde, ölüm ...","Birisi yaşadığı yerin dışında öldüğünde, ölüm ...",hukuk,Türk Medeni Kanunu,ÜÇÜNCÜ KİTAP\r\nMİRAS HUKUKU\r\nİKİNCİ KISIM\r...,8


In [11]:
df_clean = df.copy()

df_clean = df_clean.dropna(subset=["soru", "cevap", "context"])

df_clean = df_clean.drop_duplicates()

print("New shape:", df_clean.shape)

New shape: (13707, 6)


In [12]:
df_clean["soru_len"] = df_clean["soru"].astype(str).apply(len)
df_clean["cevap_len"] = df_clean["cevap"].astype(str).apply(len)
df_clean["context_len"] = df_clean["context"].astype(str).apply(len)

df_clean = df_clean[
    (df_clean["soru_len"] > 10) &
    (df_clean["cevap_len"] > 10) &
    (df_clean["context_len"] > 50)
]

print("Shape after filtering:", df_clean.shape)

Shape after filtering: (13707, 9)


In [13]:
df_clean = df_clean.drop_duplicates(subset=["soru"])

print("After question-based dedup:", df_clean.shape)

After question-based dedup: (12885, 9)


In [14]:
retrieval_df = df_clean[["context"]].copy()

retrieval_df = retrieval_df.drop_duplicates()

print("Retrieval dataset size:", retrieval_df.shape)

Retrieval dataset size: (240, 1)


In [15]:
def chunk_text(text, chunk_size=300, overlap=50):
    text = str(text)
    chunks = []
    
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
        
    return chunks

In [16]:
all_chunks = []

for text in retrieval_df["context"]:
    chunks = chunk_text(text)
    all_chunks.extend(chunks)

print("Total chunks:", len(all_chunks))

Total chunks: 5811


In [17]:
chunks_df = pd.DataFrame({"chunk_text": all_chunks})
chunks_df = chunks_df.drop_duplicates().reset_index(drop=True)

print("Unique chunks:", chunks_df.shape)
chunks_df.head()

Unique chunks: (5811, 1)


,chunk_text
0,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...
1,ılap ve ilkeleri doğrultusunda;\n\nDünya mille...
2,"lak üstünlüğü, egemenliğin kayıtsız şartsız Tü..."
3,"cağı;\n\nKuvvetler ayrımının, Devlet organları..."
4,da bulunduğu;\n\nHiçbir faaliyetin Türk milli ...


In [18]:
chunks_df["chunk_len"] = chunks_df["chunk_text"].astype(str).apply(len)
chunks_df["chunk_len"].describe()

count    5811.000000
mean      292.358458
std        38.708697
min         1.000000
25%       300.000000
50%       300.000000
75%       300.000000
max       300.000000
Name: chunk_len, dtype: float64

In [19]:
chunks_df.to_csv("../data/processed/retrieval_chunks.csv", index=False, encoding="utf-8-sig")

print("Saved!")

Saved!


In [20]:
print(chunks_df.shape)
chunks_df.head()

(5811, 2)


,chunk_text,chunk_len
0,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,300
1,ılap ve ilkeleri doğrultusunda;\n\nDünya mille...,300
2,"lak üstünlüğü, egemenliğin kayıtsız şartsız Tü...",300
3,"cağı;\n\nKuvvetler ayrımının, Devlet organları...",300
4,da bulunduğu;\n\nHiçbir faaliyetin Türk milli ...,300


In [21]:
chunks_df = pd.read_csv("../data/processed/retrieval_chunks.csv")
print(chunks_df.shape)
chunks_df.head()

(5811, 2)


,chunk_text,chunk_len
0,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,300
1,ılap ve ilkeleri doğrultusunda;\n\nDünya mille...,300
2,"lak üstünlüğü, egemenliğin kayıtsız şartsız Tü...",300
3,"cağı;\n\nKuvvetler ayrımının, Devlet organları...",300
4,da bulunduğu;\n\nHiçbir faaliyetin Türk milli ...,300


In [22]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

c:\Users\Gaming\Desktop\turkish-legal-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Gaming\Desktop\turkish-legal-rag\.venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Gaming\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode 

In [23]:
sample_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].head(5).tolist(),
    show_progress_bar=True
)

print(sample_embeddings.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.65it/s]

(5, 384)


In [24]:
chunk_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", chunk_embeddings.shape)

Batches: 100%|██████████| 182/182 [02:13<00:00,  1.37it/s]

Embeddings shape: (5811, 384)


In [25]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index total vectors:", index.ntotal)

FAISS index total vectors: 5811


In [26]:
query = "Türkiye Cumhuriyetinin yönetim şekli nedir?"
query_embedding = embedding_model.encode([query], convert_to_numpy=True)

k = 5
distances, indices = index.search(query_embedding, k)

print("Top 5 retrieved chunks:\n")

for rank, idx in enumerate(indices[0], start=1):
    print(f"{rank}. Sonuç:\n")
    print(chunks_df.iloc[idx]["chunk_text"])
    print("\n" + "-" * 100 + "\n")

Top 5 retrieved chunks:

1. Sonuç:

ve tarih huzurunda, namusum ve şerefim üzerine andiçerim."

 

D. Görev ve yetkileri

Madde 104  – (Değişik: 21/1/2017-6771/8 md.)

Cumhurbaşkanı Devletin başıdır. Yürütme yetkisi Cumhurbaşkanına aittir.

Cumhurbaşkanı, Devlet başkanı sıfatıyla Türkiye Cumhuriyetini ve Türk Milletinin birliğini tems

----------------------------------------------------------------------------------------------------

2. Sonuç:

ÜÇÜNCÜ KISIM

CUMHURİYETİN TEMEL ORGANLARI

BİRİNCİ BÖLÜM

Yasama

Öncesi…

II. Türkiye Büyük Millet Meclisinin görev ve yetkileri

 

A. Genel olarak

Madde 87 – (Değişik: 21/1/2017-6771/5 md.)

Türkiye Büyük Millet Meclisinin görev ve yetkileri, kanun koymak, değiştirmek ve kaldırmak; bütçe ve kes

----------------------------------------------------------------------------------------------------

3. Sonuç:

BAŞLANGIÇ [5]

 

Türk Vatanı ve Milletinin ebedi varlığını ve Yüce Türk Devletinin bölünmez bütünlüğünü belirleyen bu Anayasa, Türkiy

In [27]:
print(indices)
print(indices.shape)

[[ 57  10   0 318   1]]
(1, 5)


In [28]:
def retrieve_top_k(query, model, index, chunks_df, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
            "distance": float(dist)
        })
    return results

In [29]:
results = retrieve_top_k(
    "Egemenlik kime aittir?",
    embedding_model,
    index,
    chunks_df,
    k=3
)

for i, item in enumerate(results, 1):
    print(f"{i}. Distance: {item['distance']}")
    print(item["chunk_text"])
    print("-" * 100)

1. Distance: 11.892723083496094
lak üstünlüğü, egemenliğin kayıtsız şartsız Türk Milletine ait olduğu ve bunu millet adına kullanmaya yetkili kılınan hiçbir kişi ve kuruluşun, bu Anayasada gösterilen hürriyetçi demokrasi ve bunun icaplarıyla belirlenmiş hukuk düzeni dışına çıkamayacağı;

Kuvvetler ayrımının, Devlet organları arası
----------------------------------------------------------------------------------------------------
2. Distance: 12.940750122070312

kendi başına yönetmek veya bunun için temsilci atamak gücünden yoksunsa,
3. Bir terekede mirasçılık hakları henüz belli değilse veya ceninin menfaatleri gerekli kılarsa, 
4. Bir tüzel kişi gerekli organlardan yoksun kalmış ve yönetimi başka yoldan 
sağlanamamışsa,
5. Bir hayır işi veya genel ya
----------------------------------------------------------------------------------------------------
3. Distance: 13.368047714233398
İKİNCİ KISIM

TEMEL HAKLAR VE ÖDEVLER

 

BİRİNCİ BÖLÜM

Genel Hükümler

 

I.  Temel hak ve hürriyetler

In [30]:
def chunk_text_smart(text, chunk_size=300, overlap=50):
    text = str(text).strip()
    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)

        if end < text_len:
            while end > start and text[end] != " ":
                end -= 1
            if end == start:
                end = min(start + chunk_size, text_len)

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        next_start = end - overlap
        if next_start <= start:
            next_start = end

        if next_start < text_len and next_start > 0:
            while next_start < text_len and text[next_start] != " ":
                next_start += 1
            while next_start < text_len and text[next_start] == " ":
                next_start += 1

        start = next_start

    return chunks

In [31]:
all_chunks = []

for text in retrieval_df["context"]:
    chunks = chunk_text_smart(text)
    all_chunks.extend(chunks)

chunks_df = pd.DataFrame({"chunk_text": all_chunks}).drop_duplicates().reset_index(drop=True)

print("Unique chunks:", chunks_df.shape)
chunks_df.head()

Unique chunks: (5965, 1)


,chunk_text
0,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...
1,ve ilkeleri doğrultusunda;\n\nDünya milletleri...
2,"üstünlüğü, egemenliğin kayıtsız şartsız Türk M..."
3,"ayrımının, Devlet organları arasında üstünlük ..."
4,"faaliyetin Türk milli menfaatlerinin, Türk var..."


In [33]:
import re

def split_long_text_safely(text, max_len=300, overlap=50):
    text = str(text).strip()
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + max_len, n)

        if end < n:
            split_pos = text.rfind(" ", start, end)
            if split_pos == -1 or split_pos <= start:
                split_pos = end
        else:
            split_pos = end

        chunk = text[start:split_pos].strip()
        if chunk:
            chunks.append(chunk)

        next_start = split_pos - overlap
        if next_start <= start:
            next_start = split_pos

        start = next_start

    return chunks


def chunk_text_by_paragraph(text, max_len=300, overlap=50, min_len=80):
    text = str(text).replace("\\n", "\n").strip()

    parts = re.split(r"\n\s*\n+", text)
    parts = [p.strip() for p in parts if p.strip()]

    merged_parts = []
    buffer = ""

    for part in parts:
        if len(buffer) == 0:
            buffer = part
        elif len(buffer) + len(part) + 2 < min_len:
            buffer += "\n\n" + part
        else:
            merged_parts.append(buffer)
            buffer = part

    if buffer:
        merged_parts.append(buffer)

    final_chunks = []

    for part in merged_parts:
        if len(part) <= max_len:
            final_chunks.append(part)
        else:
            final_chunks.extend(split_long_text_safely(part, max_len=max_len, overlap=overlap))

    return final_chunks

In [34]:
all_chunks = []

for text in retrieval_df["context"]:
    chunks = chunk_text_by_paragraph(text, max_len=300, overlap=50, min_len=80)
    all_chunks.extend(chunks)

chunks_df = pd.DataFrame({"chunk_text": all_chunks}).drop_duplicates().reset_index(drop=True)

print("Unique chunks:", chunks_df.shape)
chunks_df.head(10)

Unique chunks: (7680, 1)


,chunk_text
0,BAŞLANGIÇ [5]
1,Türk Vatanı ve Milletinin ebedi varlığını ve Y...
2,Dünya milletleri ailesinin eşit haklara sahip ...
3,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
4,"Kuvvetler ayrımının, Devlet organları arasında..."
5,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."
6,ceği ve laiklik ilkesinin gereği olarak kutsal...
7,ve politikaya kesinlikle karıştırılamayacağı; [6]
8,Her Türk vatandaşının bu Anayasadaki temel hak...
9,Topluca Türk vatandaşlarının milli gurur ve if...


In [35]:
import re
import pandas as pd

BAD_START_WORDS = {
    "ve", "veya", "ile", "da", "de", "ama", "fakat", "ancak",
    "gibi", "çünkü", "ise", "ya", "ya da", "hem", "ki"
}

def starts_bad(text):
    text = str(text).strip().lower()
    if not text:
        return True
    
    first_words = text.split()[:2]
    if not first_words:
        return True
    
    first_word = first_words[0]
    first_two = " ".join(first_words[:2])
    
    return first_word in BAD_START_WORDS or first_two in BAD_START_WORDS

def clean_chunk_boundaries(chunks, min_len=80, max_len=450):
    cleaned = []
    
    for chunk in chunks:
        chunk = str(chunk).strip()
        if not chunk:
            continue
        
        if cleaned and (len(chunk) < min_len or starts_bad(chunk)):
            merged = cleaned[-1].rstrip() + " " + chunk.lstrip()
            if len(merged) <= max_len:
                cleaned[-1] = merged
            else:
                cleaned.append(chunk)
        else:
            cleaned.append(chunk)
    
    return cleaned

In [36]:
all_chunks = []

for text in retrieval_df["context"]:
    raw_chunks = chunk_text_by_paragraph(text, max_len=300, overlap=50, min_len=80)
    fixed_chunks = clean_chunk_boundaries(raw_chunks, min_len=80, max_len=450)
    all_chunks.extend(fixed_chunks)

chunks_df = pd.DataFrame({"chunk_text": all_chunks}).drop_duplicates().reset_index(drop=True)

print("Unique chunks after boundary cleaning:", chunks_df.shape)
chunks_df.head(15)

Unique chunks after boundary cleaning: (6196, 1)


,chunk_text
0,BAŞLANGIÇ [5]
1,Türk Vatanı ve Milletinin ebedi varlığını ve Y...
2,Dünya milletleri ailesinin eşit haklara sahip ...
3,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
4,"Kuvvetler ayrımının, Devlet organları arasında..."
5,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."
6,ceği ve laiklik ilkesinin gereği olarak kutsal...
7,Her Türk vatandaşının bu Anayasadaki temel hak...
8,Topluca Türk vatandaşlarının milli gurur ve if...
9,arşılıklı içten sevgi ve kardeşlik duygularıyl...


In [37]:
import re
import pandas as pd

In [38]:
def normalize_legal_text(text):
    text = str(text)

    text = text.replace("\\n", "\n")

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r" *\n *", "\n", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [39]:
def split_legal_text_into_blocks(text):
    text = normalize_legal_text(text)

    parts = re.split(r"\n\s*\n", text)

    cleaned_parts = []
    for part in parts:
        part = part.strip()
        if not part:
            continue

        cleaned_parts.append(part)

    return cleaned_parts

In [40]:
def split_long_block(block, max_len=500):
    block = block.strip()

    if len(block) <= max_len:
        return [block]

    sentences = re.split(r"(?<=[.!?;:])\s+", block)

    chunks = []
    current = ""

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        if len(current) + len(sent) + 1 <= max_len:
            current = f"{current} {sent}".strip()
        else:
            if current:
                chunks.append(current)

            if len(sent) > max_len:
                words = sent.split()
                piece = ""
                for w in words:
                    if len(piece) + len(w) + 1 <= max_len:
                        piece = f"{piece} {w}".strip()
                    else:
                        if piece:
                            chunks.append(piece)
                        piece = w
                if piece:
                    current = piece
                else:
                    current = ""
            else:
                current = sent

    if current:
        chunks.append(current)

    return chunks

In [41]:
def build_legal_chunks(text, min_len=80, max_len=500):
    blocks = split_legal_text_into_blocks(text)

    merged_blocks = []
    buffer = ""

    for block in blocks:
        if not buffer:
            buffer = block
        elif len(buffer) < min_len:
            buffer = f"{buffer}\n\n{block}".strip()
        else:
            merged_blocks.append(buffer)
            buffer = block

    if buffer:
        merged_blocks.append(buffer)

    final_chunks = []
    for block in merged_blocks:
        final_chunks.extend(split_long_block(block, max_len=max_len))

    final_chunks = [c.strip() for c in final_chunks if c and len(c.strip()) >= 30]

    return final_chunks

In [42]:
all_chunks = []

for text in retrieval_df["context"]:
    chunks = build_legal_chunks(text, min_len=80, max_len=500)
    all_chunks.extend(chunks)

chunks_df = pd.DataFrame({"chunk_text": all_chunks})
chunks_df = chunks_df.drop_duplicates().reset_index(drop=True)

print("Unique chunks:", chunks_df.shape)
chunks_df.head(20)

Unique chunks: (4470, 1)


,chunk_text
0,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
1,Dünya milletleri ailesinin eşit haklara sahip ...
2,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
3,"Kuvvetler ayrımının, Devlet organları arasında..."
4,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."
5,Her Türk vatandaşının bu Anayasadaki temel hak...
6,Topluca Türk vatandaşlarının milli gurur ve if...
7,"FİKİR, İNANÇ VE KARARIYLA anlaşılmak, sözüne v..."
8,"TÜRK MİLLETİ TARAFINDAN, demokrasiye aşık Türk..."
9,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...


In [43]:
def looks_like_bad_start(text):
    text = str(text).strip()
    if not text:
        return True

    bad_prefixes = (
        "ve ", "veya ", "ile ", "da ", "de ", "ama ", "ancak ", "fakat ",
        "gibi ", "çünkü ", "ise ", "ceği ", "acağı ", "ını ", "ini ",
        "unu ", "ünü ", "arı ", "eri ", "arşılıklı "
    )

    lowered = text.lower()
    return lowered.startswith(bad_prefixes)

bad_chunks = chunks_df[chunks_df["chunk_text"].apply(looks_like_bad_start)]
print("Bad-looking chunk count:", len(bad_chunks))
bad_chunks.head(20)

Bad-looking chunk count: 10


,chunk_text
65,ve üzerime aldığım görevi tarafsızlıkla yerine...
2841,ancak yapılan ödemeler de geri istenemez. II. ...
3400,Ancak vekile\r\nyetki verildiği veya durumun z...
3467,"Ancak saklayan, öngörülemeyen durumlar dolayıs..."
3480,"Ancak işletenler, zararın saklatan veya ziyare..."
4188,"Ancak bu ceza, dörtte birinden dörtte üçüne ka..."
4257,"Ancak bu süreler, iklim, mevsim, o yerdeki gel..."
4409,Ancak iş arama iznini toplu kullanmak isteyen ...
4434,Ancak iş sözleşmesi belirli süreli olarak yapı...
4447,veya işçilerin toplu bulunduğu yerler gibi işç...


In [44]:
chunks_df = chunks_df[~chunks_df["chunk_text"].apply(looks_like_bad_start)].reset_index(drop=True)

print("Final clean chunks:", chunks_df.shape)
chunks_df.head(10)

Final clean chunks: (4460, 1)


,chunk_text
0,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
1,Dünya milletleri ailesinin eşit haklara sahip ...
2,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
3,"Kuvvetler ayrımının, Devlet organları arasında..."
4,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."
5,Her Türk vatandaşının bu Anayasadaki temel hak...
6,Topluca Türk vatandaşlarının milli gurur ve if...
7,"FİKİR, İNANÇ VE KARARIYLA anlaşılmak, sözüne v..."
8,"TÜRK MİLLETİ TARAFINDAN, demokrasiye aşık Türk..."
9,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...


In [45]:
chunks_df.to_csv("../data/processed/retrieval_chunks.csv", index=False, encoding="utf-8-sig")
print("Clean retrieval chunks saved.")

Clean retrieval chunks saved.


In [46]:
import pandas as pd

chunks_df = pd.read_csv("../data/processed/retrieval_chunks.csv")
print("Loaded chunks:", chunks_df.shape)
chunks_df.head()

Loaded chunks: (4460, 1)


,chunk_text
0,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
1,Dünya milletleri ailesinin eşit haklara sahip ...
2,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
3,"Kuvvetler ayrımının, Devlet organları arasında..."
4,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."


In [47]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4760.52it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [48]:
sample_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].head(5).tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Sample embeddings shape:", sample_embeddings.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.95it/s]

Sample embeddings shape: (5, 384)


In [49]:
chunk_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", chunk_embeddings.shape)

Batches: 100%|██████████| 140/140 [01:46<00:00,  1.32it/s]

Embeddings shape: (4460, 384)


In [50]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index total vectors:", index.ntotal)

FAISS index total vectors: 4460


In [51]:
def retrieve_top_k(query, model, index, chunks_df, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
            "distance": float(dist)
        })
    return results

In [52]:
queries = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanının görev ve yetkileri nelerdir?"
]

for q in queries:
    print(f"\nSORGUN: {q}\n")
    results = retrieve_top_k(q, embedding_model, index, chunks_df, k=3)

    for i, item in enumerate(results, 1):
        print(f"{i}. Distance: {item['distance']:.4f}")
        print(item["chunk_text"][:500])
        print("-" * 100)


SORGUN: Egemenlik kime aittir?

1. Distance: 11.4850
Millet iradesinin mutlak üstünlüğü, egemenliğin kayıtsız şartsız Türk Milletine ait olduğu ve bunu millet adına kullanmaya yetkili kılınan hiçbir kişi ve kuruluşun, bu Anayasada gösterilen hürriyetçi demokrasi ve bunun icaplarıyla belirlenmiş hukuk düzeni dışına çıkamayacağı;
----------------------------------------------------------------------------------------------------
2. Distance: 12.6890
DÖRDÜNCÜ KİTAP
EŞYA HUKUKU
İKİNCİ KISIM
SINIRLI AYNÎ HAKLAR
BİRİNCİ BÖLÜM
İRTİFAK HAKLARI VE TAŞINMAZ YÜKÜ
ÜÇÜNCÜ AYIRIM
TAŞINMAZ YÜKÜ
A. Konusu
Madde 839- Taşınmaz yükü, bir taşınmazın malikini yalnız o taşınmazla sorumlu olmak 
üzere diğer bir kimseye bir şey vermek veya yapmakla yükümlü kılar. Hak sahibi olarak, bir başka taşınmazın maliki de gösterilebilir.
----------------------------------------------------------------------------------------------------
3. Distance: 13.5992
Tescil ve ilân Cumhurbaşkanınca çıkarılan yönetmelik hükümler

In [53]:
faiss.write_index(index, "../data/processed/faiss_index.index")
np.save("../data/processed/chunk_embeddings.npy", chunk_embeddings)

print("FAISS index and embeddings saved.")

FAISS index and embeddings saved.


In [54]:
test_queries = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanının görevleri nelerdir?"
]

for q in test_queries:
    print(f"\nQUERY: {q}\n")
    
    results = retrieve_top_k(q, embedding_model, index, chunks_df, k=3)
    
    for i, item in enumerate(results, 1):
        print(f"{i}. Distance: {item['distance']:.4f}")
        print(item["chunk_text"][:200])
        print("-" * 80)


QUERY: Egemenlik kime aittir?

1. Distance: 11.4850
Millet iradesinin mutlak üstünlüğü, egemenliğin kayıtsız şartsız Türk Milletine ait olduğu ve bunu millet adına kullanmaya yetkili kılınan hiçbir kişi ve kuruluşun, bu Anayasada gösterilen hürriyetçi 
--------------------------------------------------------------------------------
2. Distance: 12.6890
DÖRDÜNCÜ KİTAP
EŞYA HUKUKU
İKİNCİ KISIM
SINIRLI AYNÎ HAKLAR
BİRİNCİ BÖLÜM
İRTİFAK HAKLARI VE TAŞINMAZ YÜKÜ
ÜÇÜNCÜ AYIRIM
TAŞINMAZ YÜKÜ
A. Konusu
Madde 839- Taşınmaz yükü, bir taşınmazın malik
--------------------------------------------------------------------------------
3. Distance: 13.5992
Tescil ve ilân Cumhurbaşkanınca çıkarılan yönetmelik hükümlerine göre yapılır.12
V. Mal ve hakların kazanılması ve sorumluluk
Madde 105- Özgülenen malların mülkiyeti ile haklar, tüzel kişiliğin kaza
--------------------------------------------------------------------------------

QUERY: Türkiye Cumhuriyetinin yönetim şekli nedir?

1. Distance: 12.1